In [1]:
import torch, os, time
import numpy as np
from tqdm import tqdm
from vggt.vggt.models.vggt import VGGT
from vggt.vggt.utils.load_fn import load_and_preprocess_images
from vggt.vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.vggt.utils.geometry import unproject_depth_map_to_point_map
from vggt.visual_util import predictions_to_glb

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)

In [3]:
def extend_to_homogeneous(extrinsic):
    """
    Convert a 3x4 extrinsics matrix to a 4x4 homogeneous transformation matrix.
    
    Parameters:
        extrinsic (np.ndarray): A 3x4 matrix.
        
    Returns:
        np.ndarray: A 4x4 homogeneous transformation matrix.
    """
    bottom_row = np.array([[0, 0, 0, 1]])
    return np.vstack([extrinsic, bottom_row])

def invert_homogeneous(T):
    """
    Invert a 4x4 homogeneous transformation matrix.
    For T = [R | t], where T = [[R, t], [0, 1]],
    the inverse is T_inv = [[R.T, -R.T * t], [0, 1]]
    
    Parameters:
        T (np.ndarray): A 4x4 homogeneous transformation matrix.
        
    Returns:
        np.ndarray: The inverse of T.
    """
    R = T[:3, :3]
    t = T[:3, 3]
    R_inv = R.T
    t_inv = -R_inv @ t
    T_inv = np.eye(4)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv
    return T_inv

def transform_points(point_map, transformation_matrix):
    point_map = np.squeeze(point_map, axis=0)
    n_views, h, w, _ = point_map.shape
    points = point_map.reshape(-1, 3)
    ones = np.ones((points.shape[0], 1))
    points_hom = np.hstack((points, ones))
    points_transformed_hom = points_hom.dot(transformation_matrix.T)
    points_transformed = points_transformed_hom[:, :3]
    point_map_transformed = points_transformed.reshape(n_views, h, w, 3)
    return point_map_transformed

def transform_camera(extrinsics, transformation_matrix):
    new_extrinsics_list = []

    # Loop over the cameras:
    for i in range(extrinsics.shape[1]):
        # Extract the i-th camera extrinsics (shape: (3,4))
        cam_extrinsic = extrinsics[0, i]
        # Extend to 4x4
        cam_matrix = extend_to_homogeneous(cam_extrinsic)
        # Apply the inverse transformation of the origin camera.
        new_cam_matrix = transformation_matrix @ cam_matrix
        # Convert back to 3x4 by selecting the first three rows.
        new_extrinsics_list.append(new_cam_matrix[:3, :])

    # Stack the transformed extrinsics into a single array of shape (cams, 3, 4)
    new_extrinsics = np.stack(new_extrinsics_list, axis=0)

    # Add a leading dimension to recover shape (1, cams, 3, 4)
    new_extrinsics = np.expand_dims(new_extrinsics, axis=0)
    return new_extrinsics


def predict_glb(image_names, transformation_matrix, data, index): # trimesh scene, transformation of last image
    images = load_and_preprocess_images(image_names).to(device)

    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=dtype):
            b_images = images[None]  # add batch dimension
            aggregated_tokens_list, ps_idx = model.aggregator(b_images)
                    
        # Predict Cameras
        pose_enc = model.camera_head(aggregated_tokens_list)[-1]

        # Extrinsic and intrinsic matrices, following OpenCV convention (camera from world)
        extrinsic, _ = pose_encoding_to_extri_intri(pose_enc, b_images.shape[-2:])
        
        # Predict Point Maps
        point_map, point_conf = model.point_head(aggregated_tokens_list, b_images, ps_idx)

        point_map, extrinsic = point_map.cpu(), extrinsic.cpu()

        # Transform the point map and extrinsics
        inverse_extrinsic = np.linalg.inv(extend_to_homogeneous(extrinsic[0, 0]))
        extrinsic_transformation = transformation_matrix @ inverse_extrinsic
        point_transformation = np.linalg.inv(extrinsic_transformation)

        point_map = transform_points(point_map, point_transformation)
        extrinsic = transform_camera(extrinsic, extrinsic_transformation)
        new_transformation = extend_to_homogeneous(extrinsic[0, 1])

        if data['point_map'] is not None:
            point_map = np.concatenate((data['point_map'], np.expand_dims(point_map[1], axis=0)), axis=0)
        else:
            point_map = np.expand_dims(point_map[1], axis=0)
        if data['extrinsic'] is not None:
            extrinsic = np.concatenate((data['extrinsic'], np.expand_dims(extrinsic[0, 1], axis=0)), axis=0)
        else:
            extrinsic = np.expand_dims(extrinsic[0, 1], axis=0)

        if data['point_conf'] is not None:
            point_conf = np.concatenate((data['point_conf'], np.expand_dims(point_conf.cpu()[0, 1], axis=0)), axis=0)
        else:
            point_conf = np.expand_dims(point_conf.cpu()[0, 1], axis=0)
        if data['images'] is not None:
            images = np.concatenate((data['images'], np.expand_dims(images.cpu().numpy()[1], axis=0)), axis=0)
        else:
            images = np.expand_dims(images.cpu().numpy()[1], axis=0)
        
        return point_map, point_conf, images, extrinsic, new_transformation

In [19]:
frame_names = ['apartment_images/' + frame for frame in sorted(os.listdir('apartment_images'))][:200:15]

transformation = np.eye(4)

point_map, point_conf, images, extrinsic = None, None, None, None

for i in tqdm(range(2, len(frame_names))):
    point_map, point_conf, images, extrinsic, transformation = predict_glb(frame_names[i-2:i+1], transformation, {
        "point_map": point_map,
        "point_conf": point_conf,
        "images": images,
        "extrinsic": extrinsic
    }, i-2)

args = {
    "world_points": point_map,
    "world_points_conf": point_conf,
    "images": images,
    "extrinsic": extrinsic
}

predictions_map = predictions_to_glb(args)

100%|██████████| 12/12 [00:02<00:00,  4.09it/s]


In [ ]:
predictions_map.show(background=[0, 0, 0, 255])

In [17]:
frame_names = ['apartment_images/' + frame for frame in sorted(os.listdir('apartment_images'))][:200:15]
images = load_and_preprocess_images(frame_names).to(device)

with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        b_images = images[None]  # add batch dimension
        aggregated_tokens_list, ps_idx = model.aggregator(b_images)
                
    # Predict Cameras
    pose_enc = model.camera_head(aggregated_tokens_list)[-1]

    # Extrinsic and intrinsic matrices, following OpenCV convention (camera from world)
    extrinsic, _ = pose_encoding_to_extri_intri(pose_enc, b_images.shape[-2:])

    # Predict Point Maps
    point_map, point_conf = model.point_head(aggregated_tokens_list, b_images, ps_idx)

args = {
    "world_points": point_map.cpu(),
    "world_points_conf": point_conf.cpu(),
    "images": images.cpu().numpy(),
    "extrinsic": extrinsic.squeeze(0).cpu()
}

predictions_map = predictions_to_glb(args)

In [ ]:
predictions_map.show(background=[0, 0, 0, 255])